In [1]:
from pathlib import Path
import pandas as pd

# Ajusta la carpeta si tu ruta es otra
CARPETA = Path(r"C:\Users\luisf\IQ Tech\DashboardRotacion\Stock Odoo Cuatitlan")

# Opción recomendada: salida del segundo script
ARCHIVO_FINAL = CARPETA / "base_dashboard_rotacion.xlsx"
HOJA_FINAL = "dashboard_ventas_canal"

# Opción alternativa: salida del primer script
ARCHIVO_RAW = CARPETA / "rotacion_inventario_base_dashboard_odoo_autoazur.xlsx"
HOJA_RAW = "ventas_conjunto_detalle"

def resumir_ventas_por_canal_desde_final(archivo: Path) -> pd.DataFrame:
    df = pd.read_excel(archivo, sheet_name=HOJA_FINAL)
    df.columns = [str(c).strip() for c in df.columns]

    # La hoja ya viene filtrada a 3 meses; aquí solo consolidamos por canal.
    if "canal" not in df.columns:
        raise KeyError("No encontré la columna 'canal' en dashboard_ventas_canal")
    if "venta_total" not in df.columns:
        raise KeyError("No encontré la columna 'venta_total' en dashboard_ventas_canal")

    for col in ["venta_total", "unidades", "pedidos"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

    resumen = (
        df.groupby("canal", as_index=False)
          .agg(
              ventas_totales=("venta_total", "sum"),
              unidades=("unidades", "sum") if "unidades" in df.columns else ("venta_total", "size"),
              pedidos=("pedidos", "sum") if "pedidos" in df.columns else ("venta_total", "size"),
          )
          .sort_values("ventas_totales", ascending=False)
          .reset_index(drop=True)
    )

    return resumen

def resumir_ventas_por_canal_desde_raw(archivo: Path) -> pd.DataFrame:
    df = pd.read_excel(archivo, sheet_name=HOJA_RAW)
    df.columns = [str(c).strip() for c in df.columns]

    for col in ["fecha", "venta_total", "cantidad"]:
        if col in df.columns:
            if col == "fecha":
                df[col] = pd.to_datetime(df[col], errors="coerce")
            else:
                df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

    if "fecha" not in df.columns:
        raise KeyError("No encontré la columna 'fecha' en ventas_conjunto_detalle")
    if "canal" not in df.columns:
        raise KeyError("No encontré la columna 'canal' en ventas_conjunto_detalle")
    if "venta_total" not in df.columns:
        raise KeyError("No encontré la columna 'venta_total' en ventas_conjunto_detalle")

    fecha_fin = df["fecha"].max().normalize()
    fecha_inicio = fecha_fin - pd.Timedelta(days=90 - 1)

    df_3m = df[
        df["fecha"].notna()
        & (df["fecha"] >= fecha_inicio)
        & (df["fecha"] <= fecha_fin + pd.Timedelta(days=1))
    ].copy()

    resumen = (
        df_3m.groupby("canal", as_index=False)
             .agg(
                 ventas_totales=("venta_total", "sum"),
                 unidades=("cantidad", "sum") if "cantidad" in df_3m.columns else ("venta_total", "size"),
                 pedidos=("pedido", "nunique") if "pedido" in df_3m.columns else ("venta_total", "size"),
             )
             .sort_values("ventas_totales", ascending=False)
             .reset_index(drop=True)
    )

    return resumen

if ARCHIVO_FINAL.exists():
    resumen = resumir_ventas_por_canal_desde_final(ARCHIVO_FINAL)
else:
    resumen = resumir_ventas_por_canal_desde_raw(ARCHIVO_RAW)

print(resumen.to_string(index=False))

        canal  ventas_totales  unidades  pedidos
         Full     48106720.68     40601    38110
         Drop      3745593.08      1970     1889
    LIVERPOOL       407513.30       189      187
       AMAZON       356587.24       225      210
MERCADO LIBRE       144787.25        66       65
      WALMART       133224.00        41       39
      ELEKTRA        73716.00        32       31
       COPPEL         4311.00         9        9
      TIK TOK          359.00         1        1
